# Hansen Ch.23 Nonlinear Least Squares

理论逐步解答见 md（**23.1–23.10**）。本 notebook：CES / kink / 平滑门槛 NLLS。

In [ ]:

# Hansen Ch.23 NLLS — 详尽注释
import numpy as np
import pandas as pd
from pathlib import Path
from numpy.linalg import pinv
from scipy.optimize import minimize

ROOT = Path("/home/fang/Project/zhihu-paper/p1/hansen/econometrics/data")


def ces_mean(theta, X1, X2):
    """CES: log Y = beta + (nu/rho)*log(alpha X1^rho + (1-alpha) X2^rho)."""
    rho, nu, alpha, beta = theta
    alpha = np.clip(alpha, 1e-6, 1 - 1e-6)
    rho = np.clip(rho, -10, 0.999)
    if abs(rho) < 1e-6:
        # Cobb-Douglas 极限
        return beta + nu * (alpha * np.log(X1) + (1 - alpha) * np.log(X2))
    inside = np.maximum(alpha * (X1 ** rho) + (1 - alpha) * (X2 ** rho), 1e-300)
    return beta + (nu / rho) * np.log(inside)


def nlls_ces(Y, X1, X2):
    """NLLS for CES with multiple starts + box constraints."""
    def ssr(th):
        return np.mean((Y - ces_mean(th, X1, X2)) ** 2)

    bounds = [(-5, 0.99), (0.01, 5), (0.01, 0.99), (None, None)]
    starts = [
        np.array([0.36, 1.05, 0.39, float(np.mean(Y))]),
        np.array([0.3, 1.0, 0.4, float(np.mean(Y))]),
        np.array([-0.2, 1.0, 0.5, float(np.mean(Y))]),
        np.array([0.5, 0.9, 0.3, float(np.mean(Y))]),
    ]
    best = None
    for s in starts:
        r = minimize(ssr, s, method="L-BFGS-B", bounds=bounds)
        if best is None or r.fun < best.fun:
            best = r
    return best.x, best.fun


def cluster_se_nlls(Y, mean_fn, th, clusters, *args, eps=1e-6):
    """Sandwich V with cluster meat; mean_fn(th, *args) -> fitted values."""
    mhat = mean_fn(th, *args)
    e = Y - mhat
    n, k = len(Y), len(th)
    G = np.zeros((n, k))
    for j in range(k):
        th2 = th.copy()
        th2[j] += eps
        G[:, j] = (mean_fn(th2, *args) - mhat) / eps
    Q = G.T @ G / n
    meat = np.zeros((k, k))
    for g in np.unique(clusters):
        mg = clusters == g
        s = G[mg].T @ e[mg]
        meat += np.outer(s, s)
    nG = len(np.unique(clusters))
    Omega = meat / n
    V = pinv(Q) @ Omega @ pinv(Q) / n * (nG / (nG - 1))
    se = np.sqrt(np.maximum(np.diag(V), 0.0))
    return se, V


def hc_se_nlls(Y, mean_fn, th, *args, eps=1e-5):
    mhat = mean_fn(th, *args)
    e = Y - mhat
    n, k = len(Y), len(th)
    G = np.zeros((n, k))
    for j in range(k):
        th2 = th.copy()
        th2[j] += eps
        G[:, j] = (mean_fn(th2, *args) - mhat) / eps
    Q = G.T @ G / n
    Omega = (G * e[:, None]).T @ (G * e[:, None]) / n
    V = pinv(Q) @ Omega @ pinv(Q) / n
    return np.sqrt(np.maximum(np.diag(V), 0.0)), V


## 23.8 CES with alternative capital stocks

In [ ]:
pss = pd.read_excel(ROOT / "PSS2017/PSS2017.xlsx")
Y = np.log(pd.to_numeric(pss["EG_total"], errors="coerce").values)
X1 = pd.to_numeric(pss["EC_c_alt"], errors="coerce").values
X2 = pd.to_numeric(pss["EC_d_alt"], errors="coerce").values
country = pss["country"].values
m = np.isfinite(Y) & (X1 > 0) & (X2 > 0)
Y, X1, X2, country = Y[m], X1[m], X2[m], country[m]
print("n =", len(Y), "countries =", len(np.unique(country)))

th, ssr = nlls_ces(Y, X1, X2)
se, V = cluster_se_nlls(Y, ces_mean, th, country, X1, X2)
names = ["rho", "nu", "alpha", "beta"]
print("SSR =", ssr)
for n_, b, s in zip(names, th, se):
    print(f"  {n_:6s} {b:10.4f}  ({s:.4f})")
sig = 1 / (1 - th[0])
# delta method for sigma = 1/(1-rho)
J = np.array([1 / (1 - th[0]) ** 2, 0, 0, 0])
print(f"  sigma  {sig:10.4f}  ({np.sqrt(J @ V @ J):.4f})")
print("Table 23.1 (capacity): rho=0.36, nu=1.05, alpha=0.39, sigma=1.57")


## 23.9 RR2010: inflation ~ kink in lagged debt

In [ ]:
rr = pd.read_excel(ROOT / "RR2010/RR2010.xlsx").sort_values("year")
rr["Y"] = pd.to_numeric(rr["inflation"], errors="coerce")
rr["X"] = pd.to_numeric(rr["debt"], errors="coerce")
rr["Ylag"] = rr["Y"].shift(1)
rr["Xlag"] = rr["X"].shift(1)
d = rr.dropna(subset=["Y", "Xlag", "Ylag"])
Y, X, Ylag = d["Y"].values, d["Xlag"].values, d["Ylag"].values


def kink_mean(th, X, Ylag):
    b1, b2, b3, b4, c = th
    U = X - c
    return b1 * np.minimum(U, 0) + b2 * np.maximum(U, 0) + b3 * Ylag + b4


# concentrated search over kink c
cs = np.unique(np.quantile(X, np.linspace(0.15, 0.85, 81)))
best = None
for c in cs:
    Z = np.column_stack([np.minimum(X - c, 0), np.maximum(X - c, 0), Ylag, np.ones(len(Y))])
    b = pinv(Z.T @ Z) @ (Z.T @ Y)
    ssr = np.mean((Y - Z @ b) ** 2)
    if best is None or ssr < best[0]:
        best = (ssr, c, b)
_, c0, b0 = best
th0 = np.array([b0[0], b0[1], b0[2], b0[3], c0], dtype=float)
th = minimize(lambda th: np.mean((Y - kink_mean(th, X, Ylag)) ** 2), th0, method="Nelder-Mead").x
se, _ = hc_se_nlls(Y, kink_mean, th, X, Ylag)
print("n =", len(Y))
for n_, b, s in zip(["b1", "b2", "b3", "b4", "c"], th, se):
    print(f"  {n_:4s} {b:10.4f}  ({s:.4f})")


## 23.10 Nerlove smooth threshold cost function

In [ ]:
ner = pd.read_excel(ROOT / "Nerlove1963/Nerlove1963.xlsx")
TC = pd.to_numeric(ner["Cost"], errors="coerce").values
Q = pd.to_numeric(ner["output"], errors="coerce").values
PL = pd.to_numeric(ner["Plabor"], errors="coerce").values
PK = pd.to_numeric(ner["Pcapital"], errors="coerce").values
PF = pd.to_numeric(ner["Pfuel"], errors="coerce").values
m = (TC > 0) & (Q > 0) & (PL > 0) & (PK > 0) & (PF > 0)
logTC = np.log(TC[m])
logQ = np.log(Q[m])
logP = np.log(PL[m]) + np.log(PK[m]) + np.log(PF[m])
print("n =", len(logTC))
print("logQ quantiles:", np.round(np.quantile(logQ, [0.15, 0.5, 0.85]), 3))


def smooth_mean(th, logQ, logP):
    b1, b2, b3, b4, g = th
    w = 1.0 / (1.0 + np.exp(-(logQ - g)))
    return b1 + b2 * logQ + b3 * logP + b4 * logQ * w


# concentrated over gamma then refine
gs = np.linspace(np.quantile(logQ, 0.15), np.quantile(logQ, 0.85), 120)
best = None
for g in gs:
    w = 1.0 / (1.0 + np.exp(-(logQ - g)))
    Z = np.column_stack([np.ones(len(logTC)), logQ, logP, logQ * w])
    b = pinv(Z.T @ Z) @ (Z.T @ logTC)
    ssr = np.mean((logTC - Z @ b) ** 2)
    if best is None or ssr < best[0]:
        best = (ssr, g, b)
th0 = np.array([best[2][0], best[2][1], best[2][2], best[2][3], best[1]])
th = minimize(lambda th: np.mean((logTC - smooth_mean(th, logQ, logP)) ** 2), th0, method="Nelder-Mead", options={"maxiter": 15000}).x
se, _ = hc_se_nlls(logTC, smooth_mean, th, logQ, logP)
print("estimates:")
for n_, b, s in zip(["b1", "b2", "b3", "b4", "gamma"], th, se):
    print(f"  {n_:6s} {b:10.4f}  ({s:.4f})")
print("n below/above gamma:", (logQ < th[4]).sum(), (logQ > th[4]).sum())
print("slope low Q ~", th[1], "; high Q ~", th[1] + th[3])
